# 1. Download pdbs

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
print(sys.path[-1])

/Users/vashqu/Desktop/MLtopKin/MLtopKin


In [2]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_RAW_DIR = os.path.join(BASE_DIR, "data_raw")
METADATA_DIR = os.path.join(BASE_DIR, "metadata")
DATA_DIR = os.path.join(BASE_DIR, "data")
LOGS_DIR = os.path.join(BASE_DIR, "logs")
print(BASE_DIR)
print(DATA_RAW_DIR)
print(METADATA_DIR)
print(DATA_DIR)

/Users/vashqu/Desktop/MLtopKin/MLtopKin
/Users/vashqu/Desktop/MLtopKin/MLtopKin/data_raw
/Users/vashqu/Desktop/MLtopKin/MLtopKin/metadata
/Users/vashqu/Desktop/MLtopKin/MLtopKin/data


In [3]:
kincore_db = pd.read_csv(
    os.path.join(METADATA_DIR, "kincore.tab"),
    sep="\t"
)

In [4]:
kincore_db = kincore_db[kincore_db["PDB"].str.len() < 6]

# Quality filter: only X-ray structures with a good enough resolution
MAX_RESOLUTION = 3.0
kincore_db = kincore_db[
    (kincore_db["Method"] == "XRAY")
    & (kincore_db["Resolution"] <= MAX_RESOLUTION)
]
kincore_db

,Organism,Group,Gene,UniprotID,PDB,Method,Resolution,Rfac,FreeRfac,DomainBoundary,...,DihedralLabel,ActivityLabel,Chelix-Saltbr,ActLoop,Ligand,LigandType,DFG_Phe,Phosphorylation,Mutations,ProteinName
0,Aplysia californica,CAMK,TWITCHIN,Q16980_APLCA,1KOBA,XRAY,2.3,0.200,0.282,47-302,...,ABAminus,Inactive,out-out,in-out,No_ligand,No_ligand,189,NaN,NaN,Twitchin-like protein
1,Aplysia californica,CAMK,TWITCHIN,Q16980_APLCA,1KOBB,XRAY,2.3,0.200,0.282,47-302,...,ABAminus,Inactive,out-out,in-out,No_ligand,No_ligand,189,NaN,NaN,Twitchin-like protein
2,Arabidopsis thaliana,CAMK,CIPK23,CIPKN_ARATH,4CZTA,XRAY,2.3,0.179,0.234,31-286,...,BLBminus,Inactive,out-out,out-out,CPS:61316,Type1,173,NaN,NaN,CBL-interacting serine/threonine-protein kinas...
3,Arabidopsis thaliana,CAMK,CIPK23,CIPKN_ARATH,4CZTB,XRAY,2.3,0.179,0.234,31-286,...,BLBminus,Inactive,out-out,out-out,CPS:61314,Type1,173,NaN,NaN,CBL-interacting serine/threonine-protein kinas...
4,Arabidopsis thaliana,CAMK,CIPK23,CIPKN_ARATH,4CZUA,XRAY,1.9,0.194,0.233,31-286,...,BLBminus,Inactive,out-out,out-out,CPS:61321,Type1,173,NaN,T190D,CBL-interacting serine/threonine-protein kinas...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9692,Zea mays,CMGC,ACK2,CSK2A_MAIZE,2PVMA,XRAY,2.0,0.188,0.225,34-319,...,BLAminus,Inactive,in-out,in-in,P29:60501,Type1,171,NaN,V251A,Casein kinase II subunit alpha
9693,Zea mays,CMGC,ACK2,CSK2A_MAIZE,2PVNA,XRAY,2.0,0.256,0.308,34-319,...,BLAminus,Active,in-in,in-in,P63:60501,Type1,171,NaN,V251A,Casein kinase II subunit alpha
9694,Zea mays,CMGC,ACK2,CSK2A_MAIZE,3BE9A,XRAY,2.0,0.190,0.228,34-319,...,BLAminus,Active,in-in,in-in,P04:60501,Type1,171,NaN,V251A,Casein kinase II subunit alpha
9695,Zea mays,CMGC,ACK2,CSK2A_MAIZE,1ZOGA,XRAY,2.3,0.239,0.294,34-319,...,BLAminus,Active,in-in,in-in,K37:60501 K37:60502,Type1 Type1,171,NaN,NaN,Casein kinase II subunit alpha


In [5]:
grouped = {}
for _, row in kincore_db.iterrows():
    gene = row["Gene"]
    act_lab = row["ActivityLabel"]
    if gene not in grouped.keys():
        grouped[gene] = [act_lab]
    else:
        grouped[gene].append(act_lab)


has_both = [
    gene for gene, labels in grouped.items()
    if ('Active' in labels) and ("Inactive" in labels)
]

grouped_has_both = {
    k: v for k, v in grouped.items()
    if k in has_both
}

has_both_counts = {
    k: {"Active": v.count("Active"), "Inactive": v.count("Inactive")}
    for k, v in grouped_has_both.items()
}

active = 0
inactive = 0
for k, v in has_both_counts.items():
    ac = v["Active"]
    inac = v["Inactive"]
    print(f"{k}, active: {ac}, inactive: {inac}")
    m = min([ac, inac])
    active += m
    inactive += m


CTR1, active: 2, inactive: 1
GRK1, active: 14, inactive: 5
GRK2, active: 16, inactive: 7
GRK5, active: 3, inactive: 3
PRKACA, active: 275, inactive: 80
ROCK2, active: 5, inactive: 18
CAMKII, active: 5, inactive: 3
CDPK1, active: 1, inactive: 12
CDPK, active: 4, inactive: 2
PTK2, active: 1, inactive: 66
SRC, active: 33, inactive: 70
AKT1, active: 19, inactive: 1
PRKCH, active: 2, inactive: 1
PRKCI, active: 8, inactive: 2
PRKG1, active: 9, inactive: 8
ROCK1, active: 14, inactive: 27
AURKA, active: 5, inactive: 114
CAMK1, active: 1, inactive: 1
CAMK2A, active: 16, inactive: 5
CAMK2D, active: 11, inactive: 3
CAMKK1, active: 4, inactive: 2
CASK, active: 28, inactive: 3
CHEK1, active: 156, inactive: 11
DAPK1, active: 48, inactive: 13
DAPK2, active: 32, inactive: 2
DAPK3, active: 5, inactive: 7
DCLK1, active: 7, inactive: 5
MELK, active: 25, inactive: 6
PIM1, active: 178, inactive: 8
PRKAA2, active: 1, inactive: 9
SIK3, active: 6, inactive: 13
STK40, active: 2, inactive: 1
CSNK1D, active: 111

In [6]:
print(f"Dataset will have {active} active and {inactive} inactive conformations.")

Dataset will have 1083 active and 1083 inactive conformations.


In [7]:
import random

# Random (but reproducible) choice of the structures of each gene
rng = random.Random(42)

pdbs_to_download = {}
for gene in has_both:
    # Get all entries for this gene
    gene_data = kincore_db[kincore_db["Gene"] == gene]
    
    # Get active and inactive PDBs
    active_pdbs = gene_data[gene_data["ActivityLabel"] == "Active"]["PDB"].tolist()
    inactive_pdbs = gene_data[gene_data["ActivityLabel"] == "Inactive"]["PDB"].tolist()
    
    # Determine how many to take from each
    m = min(len(active_pdbs), len(inactive_pdbs))
    
    # Store the PDBs
    pdbs_to_download[gene] = {
        "Active": rng.sample(active_pdbs, m),
        "Inactive": rng.sample(inactive_pdbs, m)
    }

pdbs_to_download

{'CTR1': {'Active': ['3PPZB'], 'Inactive': ['3P86B']},
 'GRK1': {'Active': ['3QC9C', '3C4XA', '3C4WB', '4PNIA', '3C4WA'],
  'Inactive': ['4L9IA', '4WBOD', '4WBOB', '4WBOA', '4WBOC']},
 'GRK2': {'Active': ['5HE0A',
   '5HE2A',
   '5UVCA',
   '5UUUA',
   '7K7ZA',
   '3V5WA',
   '6U7CA'],
  'Inactive': ['3CIKA', '3PSCA', '3KRWA', '3PVWA', '1OMWA', '7PWDA', '3PVUA']},
 'GRK5': {'Active': ['4TNBA', '8UAPA', '4TNDA'],
  'Inactive': ['6PJXA', '8UAQA', '4WNKA']},
 'PRKACA': {'Active': ['5M0CA',
   '4WB5A',
   '2JDTA',
   '1YDSE',
   '6C0UA',
   '1SZMA',
   '3VQHA',
   '3L9LB',
   '6SOXA',
   '1XH7A',
   '4DH3A',
   '2UW8A',
   '7Y1GA',
   '6E9LA',
   '6YPPA',
   '4UJ2A',
   '5N3TA',
   '3ZO1A',
   '1XH8A',
   '5N1OA',
   '3L9MA',
   '2VO0A',
   '4IAIA',
   '1XH5A',
   '4DFZE',
   '2UW4A',
   '6MM5E',
   '4IJ9A',
   '5N3GA',
   '1Q8UA',
   '4IAZA',
   '5N3FA',
   '6FRXA',
   '5N3KA',
   '8SF8A',
   '5N3RA',
   '5N1GA',
   '2JDVA',
   '4WB8A',
   '1Q24A',
   '3OWPA',
   '3AGLB',
   '6YOTA',
   '

In [8]:
l = [
    vv for v in pdbs_to_download.values() for vv in v.values()
]
l = [i for ii in l for i in ii]
l

['3PPZB',
 '3P86B',
 '3QC9C',
 '3C4XA',
 '3C4WB',
 '4PNIA',
 '3C4WA',
 '4L9IA',
 '4WBOD',
 '4WBOB',
 '4WBOA',
 '4WBOC',
 '5HE0A',
 '5HE2A',
 '5UVCA',
 '5UUUA',
 '7K7ZA',
 '3V5WA',
 '6U7CA',
 '3CIKA',
 '3PSCA',
 '3KRWA',
 '3PVWA',
 '1OMWA',
 '7PWDA',
 '3PVUA',
 '4TNBA',
 '8UAPA',
 '4TNDA',
 '6PJXA',
 '8UAQA',
 '4WNKA',
 '5M0CA',
 '4WB5A',
 '2JDTA',
 '1YDSE',
 '6C0UA',
 '1SZMA',
 '3VQHA',
 '3L9LB',
 '6SOXA',
 '1XH7A',
 '4DH3A',
 '2UW8A',
 '7Y1GA',
 '6E9LA',
 '6YPPA',
 '4UJ2A',
 '5N3TA',
 '3ZO1A',
 '1XH8A',
 '5N1OA',
 '3L9MA',
 '2VO0A',
 '4IAIA',
 '1XH5A',
 '4DFZE',
 '2UW4A',
 '6MM5E',
 '4IJ9A',
 '5N3GA',
 '1Q8UA',
 '4IAZA',
 '5N3FA',
 '6FRXA',
 '5N3KA',
 '8SF8A',
 '5N3RA',
 '5N1GA',
 '2JDVA',
 '4WB8A',
 '1Q24A',
 '3OWPA',
 '3AGLB',
 '6YOTA',
 '1VEBA',
 '7AXTA',
 '2ERZE',
 '3DNEA',
 '6SPMA',
 '4UJ9A',
 '2UW7A',
 '4HPUE',
 '5N36A',
 '3X2UA',
 '4C36A',
 '5N3QA',
 '3O7LB',
 '6Y89A',
 '2UVZA',
 '3L9LA',
 '3E8EE',
 '3MVJE',
 '3L9MB',
 '1Q8TA',
 '2UW3A',
 '4WB6B',
 '2GNJA',
 '8FECA',
 '3E8CF',


In [9]:
with open(os.path.join(
    METADATA_DIR,
    "pdb_ids.txt"
    ), "w") as fp:
    for i, id in enumerate(l):
        if i < len(l) - 1:
            fp.write(f"{id[:4]}, ")
        else:
            fp.write(f"{id[:4]}")

# Table of the selected chains. The features are built only from these.
kincore_db[kincore_db["PDB"].isin(l)].to_csv(
    os.path.join(METADATA_DIR, "selected_chains.tab"),
    sep="\t",
    index=False
)

# 2. Clean pdbs

In [10]:
with open(os.path.join(LOGS_DIR, "cif_download.log"), "r") as r:
    lines = r.readlines()

fails = [line for line in lines if "Failed" in line]
fails

[]

### Download the failed files manually, then continue to the cleaning step below.

In [11]:
# NOTE: This was generated by Gemini 2.5 Pro, sorryyyy

import os
import gzip
from Bio.PDB import MMCIFParser, MMCIFIO, Select
from Bio.PDB.PDBExceptions import PDBConstructionWarning
import warnings

# Assume DATA_RAW_DIR, DATA_DIR and pdbs_to_download are defined
# For example:
# DATA_RAW_DIR = "path/to/your/DATA_RAW_DIR"
# DATA_DIR = "path/to/your/DATA_DIR"
# pdbs_to_download = {
# 'CTR1': {'Active': ['3PPZB'], 'Inactive': ['3P86B']},
# 'GRK1': {'Active': ['4L9IB',   '3T8OA',   '3C4WA',   '3C4WB',   '3C4XA',   '3C4XB',   '3C4ZA',   '3C50A'],
# 'Inactive': ['3C4YA',   '4L9IA',   '4WBOA',   '4WBOB',   '4WBOC',   '4WBOD',   '7MT9G',   '7MTAG']}
# }

# Custom Select class to filter for a specific chain
class ChainSelect(Select):
    """
    Custom Bio.PDB.Select class to select a specific chain.
    """
    def __init__(self, chain_id_to_select):
        # Ensure chain ID is uppercase for consistent matching
        self.chain_id_to_select = chain_id_to_select.upper()

    def accept_model(self, model):
        # Accept all models (usually there's one in crystal structures)
        return 1

    def accept_chain(self, chain):
        # Accept the chain if its ID matches the target chain ID
        if chain.get_id().upper() == self.chain_id_to_select:
            return 1
        else:
            return 0

    def accept_residue(self, residue):
        # Accept all residues within the selected chain
        return 1

    def accept_atom(self, atom):
        # Accept all atoms within the selected residues
        return 1

# Suppress common PDBConstructionWarning if they are not critical for your task.
# These warnings often relate to minor discrepancies in PDB files.
warnings.simplefilter('ignore', PDBConstructionWarning)

complete_log = ""

# Start of your existing code structure
for protein, inner_d in pdbs_to_download.items():
    t = f"Processing {protein}..."
    complete_log += t
    complete_log += "\n"
    print(t)

    prot_dir = os.path.join(DATA_DIR, protein)
    os.makedirs(prot_dir, exist_ok=True)

    for label, pdb_id_list in inner_d.items():
        t = f"\t{label} has {len(pdb_id_list)} PDBs"
        complete_log += t
        complete_log += "\n"
        print(t)

        output_dir = os.path.join(prot_dir, label)
        os.makedirs(output_dir, exist_ok=True)

        for pdb_id_entry in pdb_id_list: # e.g., "3PPZB"
            t = f"\t\tProcessing {pdb_id_entry}..."
            complete_log += t
            complete_log += "\n"
            print(t)

            # Extract PDB code (e.g., "3PPZ") and target chain ID (e.g., "B")
            target_chain_id = pdb_id_entry[-1]
            pdb_code = pdb_id_entry[:-1]

            # Construct the full path to the gzipped CIF file
            # Based on your example: "3PPZ.cif.gz" from "3PPZB"
            gz_cif_file = os.path.join(DATA_RAW_DIR, f"{pdb_code}.cif.gz")

            # Define the output file name and full path
            # Based on your example: "3PPZ_chainB.cif"
            # (mmCIF has no limits on the number of atoms/residues)
            output_pdb_filename = f"{pdb_code}_chain{target_chain_id}.cif"
            output_pdb_file_path = os.path.join(output_dir, output_pdb_filename)

            if not os.path.exists(gz_cif_file):
                t = f"\t\t\tWARNING: Source file {gz_cif_file} not found. Skipping {pdb_id_entry}."
                complete_log += t
                complete_log += "\n"
                print(t)
                
                continue

            try:
                # Initialize the MMCIFParser
                parser = MMCIFParser() # QUIET=True can be added but warnings.simplefilter is more general

                # Unzip and parse the CIF file
                # 'rt' mode for reading as text; CIF is a text-based format
                with gzip.open(gz_cif_file, 'rt', encoding='utf-8') as cif_file_handle:
                    # The first argument to get_structure is a structure ID (can be arbitrary, e.g., pdb_code)
                    structure = parser.get_structure(pdb_code, cif_file_handle)

                # Initialize MMCIFIO to write the mmCIF file
                pdb_io = MMCIFIO()
                pdb_io.set_structure(structure) # Set the full parsed structure

                # Save only the specified chain using the ChainSelect class
                pdb_io.save(output_pdb_file_path, ChainSelect(target_chain_id))

                t = f"\t\t\tSuccessfully extracted chain {target_chain_id} from {pdb_code} to {output_pdb_file_path}"
                complete_log += t
                complete_log += "\n"
                print(t)

            except FileNotFoundError: # Safeguard
                t = f"\t\t\tERROR: File {gz_cif_file} might have been removed or unzipping failed for {pdb_id_entry}. Skipping."
                complete_log += t
                complete_log += "\n"
                print(t)
            except Exception as e:
                t = f"\t\t\tERROR processing {pdb_id_entry} (from file {gz_cif_file}): {e}"
                complete_log += t
                complete_log += "\n"
                print(t)

t = "\nAll PDB processing finished."
complete_log += t
complete_log += "\n"
print(t)

Processing CTR1...
	Active has 1 PDBs
		Processing 3PPZB...


			Successfully extracted chain B from 3PPZ to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/CTR1/Active/3PPZ_chainB.cif
	Inactive has 1 PDBs
		Processing 3P86B...


			Successfully extracted chain B from 3P86 to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/CTR1/Inactive/3P86_chainB.cif
Processing GRK1...
	Active has 5 PDBs
		Processing 3QC9C...


			Successfully extracted chain C from 3QC9 to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/GRK1/Active/3QC9_chainC.cif
		Processing 3C4XA...
			Successfully extracted chain A from 3C4X to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/GRK1/Active/3C4X_chainA.cif
		Processing 3C4WB...


			Successfully extracted chain B from 3C4W to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/GRK1/Active/3C4W_chainB.cif
		Processing 4PNIA...
			Successfully extracted chain A from 4PNI to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/GRK1/Active/4PNI_chainA.cif
		Processing 3C4WA...


			Successfully extracted chain A from 3C4W to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/GRK1/Active/3C4W_chainA.cif
	Inactive has 5 PDBs
		Processing 4L9IA...


... (1,091 outputs left out)

			Successfully extracted chain A from 1OM1 to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Active/1OM1_chainA.cif
		Processing 3PVGA...
			Successfully extracted chain A from 3PVG to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Active/3PVG_chainA.cif
		Processing 1JAMA...


			Successfully extracted chain A from 1JAM to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Active/1JAM_chainA.cif
		Processing 1LR4A...
			Successfully extracted chain A from 1LR4 to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Active/1LR4_chainA.cif
	Inactive has 7 PDBs
		Processing 2OXYB...
			Successfully extracted chain B from 2OXY to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Inactive/2OXY_chainB.cif
		Processing 2PVJA...


			Successfully extracted chain A from 2PVJ to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Inactive/2PVJ_chainA.cif
		Processing 2PVMA...
			Successfully extracted chain A from 2PVM to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Inactive/2PVM_chainA.cif
		Processing 1F0QA...
			Successfully extracted chain A from 1F0Q to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Inactive/1F0Q_chainA.cif
		Processing 1DAYA...


			Successfully extracted chain A from 1DAY to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Inactive/1DAY_chainA.cif
		Processing 2PVKA...
			Successfully extracted chain A from 2PVK to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Inactive/2PVK_chainA.cif
		Processing 1ZOHA...
			Successfully extracted chain A from 1ZOH to /Users/vashqu/Desktop/MLtopKin/MLtopKin/data/ACK2/Inactive/1ZOH_chainA.cif

All PDB processing finished.


In [12]:
with open(os.path.join(LOGS_DIR, "create_pdbs.log"), "w") as log_file:
    log_file.write(complete_log)